In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import before_agent
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage
from langchain.messages import HumanMessage
from langchain.messages import SystemMessage
import re

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [6]:

from langchain.agents.middleware import before_agent

forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"],
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰"],
    "harmful": ["담배", "술", "폭력", "바보", "멍청이"],
}


@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None


In [7]:
@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None

    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3단계: 교정 (Correction / Regeneration)
    if "LEAKED" in result.content:

        # 원래 사용자의 질문을 가져오기 (문맥 파악용) -> state["messages"][-2]가 보통 사용자 질문
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        # 교정 모델에게 "정답을 빼고 힌트로 바꿔라"고 지시
        correction_prompt = f"""
        당신은 친절한 AI 튜터입니다.

        절대 정답을 직접 말하지 말고, 학생이 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
        말투는 친절하게 해주세요.

        사용자 질문: {original_question}
        """

        # LLM을 다시 호출하여 새로운 답변 생성 (비용은 1회 더 발생하지만 품질 확보)
        corrected_response = model.invoke([
            SystemMessage(content="당신은 소크라테스식 교육법을 사용하는 튜터입니다."),
            HumanMessage(content=correction_prompt)
        ])

        # 원래의 유출된 답변을 교정된 답변으로 덮어쓰기
        last_message.content = corrected_response.content

    return None


In [8]:
@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None

    last_message = state["messages"][-1]
    
    if last_message.type != "human": return None 
    # not isinstance(last_message, HumanMessage) 방식도 사용 가능
    # before_agent 훅에서 어차피 마지막 메세지는 HumanMessage 타입이기 때문에, 굳이 타입 검사를 할 필요가 없습니다.

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content): # content 안에 phone_pattern(전화번호 패턴)이 있는지 검사합니다.
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content) # 패턴이 발견되면, 그 부분을 "<PHONE_REDACTED>"라는 문구로 교체(Substitute)합니다.
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None


In [9]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]


@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None

    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None


In [10]:
# 4중 방어막이 적용된 에이전트
agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[
        education_guardrail,             # Layer 1: 입력 필터 (규칙 - 딴짓/부정행위)
        student_safety_middleware,       # Layer 2: 개인정보 보호 (전화번호 마스킹)
        counseling_escalation_middleware,# Layer 3: 상담 이관 (휴먼 에스컬레이션)
        answer_leakage_guardrail         # Layer 4: 출력 필터 (모델 기반 교정)
    ],
)


In [11]:
agent.invoke({
    "messages": [{"role": "user", "content": "저 수학 과외 구하고 싶어요. 제 번호 010-1234-5678로 연락 주세요."}]
})


🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.
원본: 저 수학 과외 구하고 싶어요. 제 번호 010-1234-5678로 연락 주세요.
수정: 저 수학 과외 구하고 싶어요. 제 번호 <PHONE_REDACTED>로 연락 주세요.


{'messages': [HumanMessage(content='저 수학 과외 구하고 싶어요. 제 번호 <PHONE_REDACTED>로 연락 주세요.', additional_kwargs={}, response_metadata={}, id='39883bdc-ce92-4c54-8f95-6dc01f0ef7fe'),
  AIMessage(content='도와드릴게요! 다만 제가 직접 전화로 연락을 대신할 수는 없어요. 대신 수학 과외를 찾는 데 바로 쓸 수 있는 정보 정리 방법과 메시지 템플릿을 드릴게요. 지역, 학년, 예산 등 정보를 알려주시면 제가 맞춤 템플릿도 바로 만들어 드립니다.\n\n먼저 정리할 정보\n- 학년/수준: 예) 초등 5학년 수준, 중3 미적분 기본 다지기 등\n- 과목 구체성: 수학 전과목, 또는 대수/기하/확률 등 특정 영역\n- 수업 형태: 온라인/오프라인(직접 만나기) 여부\n- 주당 가능 시간과 수업 길이: 예) 주 2회, 각 60분\n- 예산: 시간당 원하는 금액대\n- 지역 또는 온라인 가능 여부\n- 목표: 성적 향상, 특정 시험 대비, 기초 체력 다지기 등\n- 면담/시범수업 여부 선호\n- 연락 방법: 전화/카톡/이메일 중 선호\n\n수험생 구인/구직 채널 아이디어\n- 지역 커뮤니티 카페, 학원 커뮤니티, 네이버 카페 등에 구인 글 올리기\n- 당근마켓/헬프센터 같은 지역 플랫폼에 올리기\n- 온라인 튜터 플랫폼 이용: 예) 온라인 수학 튜터 찾기 가능 플랫폼 검색\n- 지인 추천 활용: 학교 선생님이나 친구의 추천\n\n바로 써먹기 좋은 템플릿\n1) 포스팅용 글(카페/당근마켓 등에 붙이는 글)\n제목: 수학 과외 선생님 구합니다(온라인/오프라인 가능)\n본문:\n- 지역/온라인 여부: [예: 서울시 강남구, 온라인 가능]\n- 학년/수준: [예: 고등학생 1학년, 미적분 및 기하 보충]\n- 과목: 수학(대수, 기하, 확률 등)\n- 수업 형태: [온라인/대면/혼합]\n- 주당 수업 횟수 및 시간: [예: 주 2회, 60분

In [12]:
agent.invoke({
    "messages": [{"role": "user", "content": "나 요즘 학교에서 왕따 당하는 것 같아서 너무 우울해."}]
})


✋ [상담 이관] 심각한 고민/요청 감지: 왕따


{'messages': [HumanMessage(content='나 요즘 학교에서 왕따 당하는 것 같아서 너무 우울해.', additional_kwargs={}, response_metadata={}, id='b5832f1c-632a-4dcf-8ef4-cc1cb8a88a11'),
  AIMessage(content='학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)', additional_kwargs={}, response_metadata={}, id='7716e03b-2266-475f-8208-0b0ae22fbb1b', tool_calls=[], invalid_tool_calls=[])]}

## 🛡️ `student_safety_middleware`에서 `return` 없이 `AIMessage`가 생성되는 이유

이 현상은 해당 함수가 에이전트(LLM)가 실행되기 **'전'**에 호출되는 훅(**`before_agent`**)이기 때문에 발생합니다.

### 1. `before_agent`의 역할
이 훅은 이름 그대로 **"에이전트(LLM)가 답변을 생성하기 직전"**에 실행됩니다.

1.  **사용자 입력:** 사용자가 `"수학 과외 구해요 010-1234-5678"`이라고 메시지를 보냅니다.
2.  **미들웨어 실행:** `student_safety_middleware`가 실행됩니다.
    *   여기서 `last_message.content`를 직접 수정해서 `"수학 과외 구해요 <PHONE_REDACTED>"`로 바꿉니다.
    *   `return None`을 하면 **"흐름을 끊지 말고 다음 단계로 가라"**는 뜻입니다.
3.  **에이전트(LLM) 호출:** 이제 시스템은 수정된 메시지를 가지고 실제 AI 모델(Gemini 등)을 호출합니다.
4.  **AI 답변 생성:** AI는 수정된 메시지를 보고 그에 맞는 답변(`AIMessage`)을 생성합니다.

---

### 2. 왜 `return`이 없어도 되나요?

*   **객체 참조(Reference):** 파이썬에서 `state["messages"]` 리스트 안에 들어있는 메시지 객체는 **참조 방식**입니다. 함수 안에서 `last_message.content = "..."`라고 수정하면, 함수 외부의 `state` 값도 즉시 바뀝니다.
*   **흐름의 연속성:** `before_agent`에서 아무것도 `return`하지 않거나 `None`을 반환하면, 시스템은 **"전처리가 끝났으니 이제 계획대로 AI 모델을 실행해!"**라고 판단합니다.

---

### 3. 만약 `return`을 했다면?

만약 `03-04` 예제처럼 `return {"messages": [...], "jump_to": "end"}`를 했다면 다음과 같이 동작했을 것입니다.

*   **AI 모델을 호출하지 않고** 즉시 대화를 종료합니다.
*   하지만 지금은 `return` 없이 내용만 살짝 고쳤기 때문에, AI는 **고쳐진 내용을 바탕으로 정상적으로 답변을 생성**한 것입니다.

---

> **요약:** `before_agent`에서 `return None`은 **"수정된 상태를 가지고 다음 단계(LLM 호출)로 진행하라"**는 신호입니다.